# Paper 3 Signal Comparison + Gate 1 Runner

Runs two experiments that close the main open question in Paper 3:

## Experiment 1: geometry_KCD vs semantic_KCD on the hard stress set

The key signal-comparison question: is the geometry signal specifically necessary for
constraint-critical memory tasks, or does the KCD codec structure do the work regardless of signal?

Semantic scoring cannot distinguish a user constraint turn from its assistant echo — they are
topically identical. Geometry can, because the user constraint creates a larger state displacement.

- If `geometry_KCD` beats `semantic_KCD` on hard stress → signal is doing real work
- If `semantic_KCD` ties `geometry_KCD` → codec structure was doing the work, not signal

## Experiment 2: Gate 1 — geometry refinement inside a semantic shortlist

Answers: inside a semantic shortlist, does geometry refinement improve over semantic-only compression
on real semantic-memory benchmarks (MSC valid, LongMemEval-S cleaned)?

- Oracle: Δ Kendall tau ≥ +0.03 or Δ top-5 oracle recall ≥ +5pp → geometry is adding ranking value
- Policy: `semantic_query_conditioned_geometry_KCD` vs `budget_aware_semantic_KCD`
- If it ties or loses → stop hand-designed geometry refinement, move to learned harm predictor


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR  = "/content/rt-geometry-memory"

# Model. qwen25_15b matches prior canonical artifacts.
# Switch to qwen25_05b for ~3x faster runs.
MODEL     = "qwen25_15b"

BUDGETS   = "0.20,0.35,0.50"

# Experiment 1 run name
EXP1_NAME = "paper3_signal_comparison_hardset_v1"

# ── Gate 1 scaling knobs ──────────────────────────────────────────────────────
# Oracle cost = conversations × (turns_per_conv / stride) × len(budgets) forward passes.
#
# LongMemEval conversations are 100-300 turns each. LONGMEM_MAX_TURNS caps how many
# turns are extracted per conversation — the key fix that prevents 30-min/conv runtime.
#
# Estimated T4 wall time:
#   FAST   (~1-2 h):  MSC=8,  LME=6,  STRIDE=8, MAX_T=4,  LONGMEM_MAX_TURNS=30
#   STANDARD (~3-4 h): MSC=16, LME=10, STRIDE=6, MAX_T=6,  LONGMEM_MAX_TURNS=40  ← default
#   FULL   (~6-8 h):  MSC=24, LME=12, STRIDE=4, MAX_T=8,  LONGMEM_MAX_TURNS=60

MSC_LIMIT            = 16   # MSC conversations
LONGMEM_LIMIT        = 10   # LongMemEval conversations
TARGET_STRIDE        = 6    # sample every Nth turn as a target
MAX_TARGET_TURNS     = 6    # max target turns per conversation
LONGMEM_MAX_TURNS    = 40   # truncate LongMemEval conversations to first N turns

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
print(result.stdout.strip() or "No GPU found — switch runtime to GPU in Runtime > Change runtime type")

In [ ]:
# ── Clone and install ─────────────────────────────────────────────────────────
import os
%cd /content
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!bash scripts/colab_setup.sh

In [ ]:
# ── Optional: Hugging Face login (needed for gated models) ───────────────────
# Uncomment and paste your token if model downloads fail.
# import os
# os.environ["HF_TOKEN"] = "hf_..."
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])

---
## Download and prepare benchmarks from source

Downloads MSC valid and LongMemEval-S cleaned directly from HuggingFace.
No manual upload needed. Both datasets are public and require no login token.

In [ ]:
%cd {REPO_DIR}

# ── Download MSC valid from HuggingFace ───────────────────────────────────────
MSC_RAW      = f"{REPO_DIR}/benchmarks/msc_valid_raw.jsonl"
MSC_NORM     = f"{REPO_DIR}/benchmarks/msc_valid_normalized.jsonl"

if not __import__('os').path.exists(MSC_NORM):
    print("Downloading MSC valid from HuggingFace...")
    !python scripts/download_public_benchmark.py \
        --benchmark msc_valid \
        --output "{MSC_RAW}"
    !python scripts/prepare_public_benchmark_jsonl.py \
        --format msc \
        --input  "{MSC_RAW}" \
        --output "{MSC_NORM}" \
        --family msc_valid
    print(f"MSC valid ready: {MSC_NORM}")
else:
    print(f"MSC valid already present: {MSC_NORM}")

# ── Download LongMemEval-S cleaned from HuggingFace ──────────────────────────
LONGMEM_RAW  = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_raw.json"
LONGMEM_NORM = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_normalized.jsonl"

if not __import__('os').path.exists(LONGMEM_NORM):
    print("Downloading LongMemEval-S cleaned from HuggingFace...")
    !python scripts/download_public_benchmark.py \
        --benchmark longmemeval_s_cleaned \
        --output "{LONGMEM_RAW}"
    !python scripts/prepare_public_benchmark_jsonl.py \
        --format longmemeval \
        --input  "{LONGMEM_RAW}" \
        --output "{LONGMEM_NORM}" \
        --family longmemeval_s_cleaned
    print(f"LongMemEval-S cleaned ready: {LONGMEM_NORM}")
else:
    print(f"LongMemEval-S cleaned already present: {LONGMEM_NORM}")

# ── Verify ────────────────────────────────────────────────────────────────────
import os
for label, path in [("MSC valid", MSC_NORM), ("LongMemEval-S", LONGMEM_NORM)]:
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    lines   = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    print(f"{label}: {lines} conversations, {size_mb:.1f} MB  →  {path}")

---
## Experiment 1: geometry_KCD vs semantic_KCD — Hard Stress Set

Policies: `uniform`, `semantic`, `geometry`, `geometry_keep_compress_drop`, `semantic_keep_compress_drop`

Benchmark: hard stress set (`long_dependency`, `retrieval_heavy`, `code_conversation`)

In [ ]:
%cd {REPO_DIR}
!bash scripts/run_paper3_signal_comparison_hardset.sh "{EXP1_NAME}" "{MODEL}" "{BUDGETS}"

In [ ]:
# ── Read Experiment 1 results ────────────────────────────────────────────────
%cd {REPO_DIR}
pairwise_path = f"results/paper3/studies/{EXP1_NAME}/pairwise_report.md"
if os.path.exists(pairwise_path):
    with open(pairwise_path) as f:
        print(f.read())
else:
    print("Pairwise report not found — check run output above for errors.")

In [ ]:
# ── Memory critical analysis — geometry_KCD vs semantic_KCD at budget 0.35 ───
import glob
for pattern in [f"results/paper3/studies/{EXP1_NAME}/memory_critical_*_geometry_kcd_b035.md",
                f"results/paper3/studies/{EXP1_NAME}/memory_critical_*_semantic_kcd_b035.md"]:
    for path in sorted(glob.glob(pattern)):
        print(f"\n{'='*60}\n{path}\n{'='*60}")
        with open(path) as f:
            print(f.read())

---
## Experiment 2: Gate 1 — Geometry Refinement Inside a Semantic Shortlist

Runs oracle headroom probe + policy refinement study on:
- `MSC valid` (persona / preference continuity)
- `LongMemEval-S cleaned` (episodic retrieval)

Policies: `semantic`, `budget_aware_semantic_KCD`, `semantic_ambient_geometry_KCD`, `semantic_query_conditioned_geometry_KCD`

---
## Experiment 2: Gate 1 — Geometry Refinement Inside a Semantic Shortlist

Runs oracle headroom probe + policy refinement study on:
- `MSC valid` (persona / preference continuity)
- `LongMemEval-S cleaned` (episodic retrieval)

Policies: `semantic`, `budget_aware_semantic_KCD`, `semantic_ambient_geometry_KCD`, `semantic_query_conditioned_geometry_KCD`

**Runtime note:** the oracle probe does ~`conversations × (turns/stride) × budgets` forward passes.
With the STANDARD config above this is roughly 4-5 hours total on a T4.
On an A100 expect ~2-3x faster. Results are written incrementally — a disconnect mid-run
loses only the current conversation, not prior ones.

In [ ]:
# ── Mount Google Drive for persistent result storage (optional but recommended) ─
# If you mount Drive, results are saved there and survive Colab disconnects.
# Skip this cell if you don't want to use Drive.

USE_DRIVE = True   # set False to skip

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os, shutil

    DRIVE_RESULTS = "/content/drive/MyDrive/rt_results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

    # Symlink results/ inside the repo to Drive so writes go there directly
    LOCAL_RESULTS = f"{REPO_DIR}/results"
    if not os.path.islink(LOCAL_RESULTS):
        if os.path.isdir(LOCAL_RESULTS):
            shutil.copytree(LOCAL_RESULTS, DRIVE_RESULTS, dirs_exist_ok=True)
            shutil.rmtree(LOCAL_RESULTS)
        os.symlink(DRIVE_RESULTS, LOCAL_RESULTS)
        print(f"Results will be written to Drive: {DRIVE_RESULTS}")
    else:
        print(f"Results symlink already set: {LOCAL_RESULTS} -> {os.readlink(LOCAL_RESULTS)}")
else:
    print("Skipping Drive mount — results will only be stored in /content (lost on disconnect).")

In [ ]:
# ── Reconnect recovery: restore results symlink from Drive ────────────────────
# Run this cell after a disconnect to re-attach Drive results before continuing.
# Safe to run even on a fresh session — it's a no-op if Drive isn't mounted.

import os
if os.path.exists("/content/drive/MyDrive/rt_results"):
    LOCAL_RESULTS = f"{REPO_DIR}/results"
    if not os.path.islink(LOCAL_RESULTS):
        if os.path.isdir(LOCAL_RESULTS):
            import shutil; shutil.rmtree(LOCAL_RESULTS)
        os.symlink("/content/drive/MyDrive/rt_results", LOCAL_RESULTS)
        print("Restored results symlink from Drive.")
    already_done = [
        p for p in [
            f"{LOCAL_RESULTS}/paper3/harm_oracle/paper3_gate1_oracle_msc_valid/summary.json",
            f"{LOCAL_RESULTS}/paper3/studies/paper3_gate1_refinement_msc_valid/pairwise_report.md",
            f"{LOCAL_RESULTS}/paper3/harm_oracle/paper3_gate1_oracle_longmemeval_s_cleaned/summary.json",
            f"{LOCAL_RESULTS}/paper3/studies/paper3_gate1_refinement_longmemeval_s_cleaned/pairwise_report.md",
        ] if os.path.exists(p)
    ]
    print(f"Completed outputs found on Drive ({len(already_done)}/4):")
    for p in already_done:
        print(f"  ✓ {p}")
    if len(already_done) < 4:
        missing = 4 - len(already_done)
        print(f"  {missing} output(s) still to run — continue from the Gate 1 cell below.")
else:
    print("Drive not mounted. Run the Drive mount cell first if you want persistence.")

In [ ]:
%cd {REPO_DIR}
MSC_PATH     = f"{REPO_DIR}/benchmarks/msc_valid_normalized.jsonl"
LONGMEM_PATH = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_normalized.jsonl"

!bash scripts/run_paper3_gate1_real.sh \
    "{MSC_PATH}" \
    "{LONGMEM_PATH}" \
    "{MODEL}" \
    "{BUDGETS}" \
    {MSC_LIMIT} \
    {LONGMEM_LIMIT} \
    {TARGET_STRIDE} \
    {MAX_TARGET_TURNS} \
    {LONGMEM_MAX_TURNS}

In [ ]:
# ── Read Gate 1 oracle results ────────────────────────────────────────────────
%cd {REPO_DIR}
for oracle_name in ["paper3_gate1_oracle_msc_valid", "paper3_gate1_oracle_longmemeval_s_cleaned"]:
    path = f"results/paper3/harm_oracle/{oracle_name}/oracle_summary.md"
    if os.path.exists(path):
        print(f"\n{'='*60}\nOracle: {oracle_name}\n{'='*60}")
        with open(path) as f:
            print(f.read())
    else:
        # fall back to any summary file in that directory
        for p in sorted(glob.glob(f"results/paper3/harm_oracle/{oracle_name}/*.md")):
            print(f"\n{'='*60}\n{p}\n{'='*60}")
            with open(p) as f:
                print(f.read())

In [ ]:
# ── Read Gate 1 refinement study pairwise reports ─────────────────────────────
%cd {REPO_DIR}
for study_name in ["paper3_gate1_refinement_msc_valid", "paper3_gate1_refinement_longmemeval_s_cleaned"]:
    path = f"results/paper3/studies/{study_name}/pairwise_report.md"
    if os.path.exists(path):
        print(f"\n{'='*60}\nRefinement study: {study_name}\n{'='*60}")
        with open(path) as f:
            print(f.read())
    else:
        print(f"Not found: {path}")

---
## Save and download results

Zip the full results directory and download it.

In [ ]:
%cd {REPO_DIR}
!zip -r /content/rt_signal_gate1_results.zip \
    results/paper3/studies/{EXP1_NAME} \
    results/paper3/harm_oracle/paper3_gate1_oracle_msc_valid \
    results/paper3/harm_oracle/paper3_gate1_oracle_longmemeval_s_cleaned \
    results/paper3/studies/paper3_gate1_refinement_msc_valid \
    results/paper3/studies/paper3_gate1_refinement_longmemeval_s_cleaned \
    2>/dev/null || echo "Some directories not found — only completed runs were zipped."

from google.colab import files
files.download("/content/rt_signal_gate1_results.zip")